# 💳 Dataset 6: Credit Approval — UCI
**Propósito:** Clasificación binaria — Aprobación de tarjeta de crédito.  
**Fuente:** UCI ML Repository (origen: posiblemente Japón — datos completamente anonimizados).  
> ⚠️ Todos los atributos se llaman A1–A15. Los valores categóricos son letras sin sentido semántico.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import KNNImputer
from sklearn.metrics import classification_report, roc_auc_score
import warnings
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
print("✅ Librerías cargadas")

## 1. Carga de Datos

In [ ]:
cols = [f'A{i}' for i in range(1,16)] + ['TARGET']
df = pd.read_csv('../datasets/4 data sets/dataset6/crx.data', 
                 header=None, names=cols, na_values='?')

# Renombrar target: + = aprobado (1), - = rechazado (0)
df['TARGET'] = (df['TARGET'] == '+').astype(int)

print(f"Shape: {df.shape}")
print(f"\nDistribución del target:")
print(df['TARGET'].value_counts())
df.head(3)

## 2. Análisis de Valores Nulos

In [ ]:
nulos = df.isnull().sum()
nulos_pct = (nulos/len(df)*100).round(2)
resumen = pd.DataFrame({'Nulos': nulos, '%': nulos_pct}).query('Nulos > 0')
print("Columnas con valores faltantes (? → NaN):")
print(resumen)

## 3. Identificación de Tipos de Variables

In [ ]:
cat_cols = [c for c in df.columns if df[c].dtype == 'object']
num_cols = [c for c in df.columns if df[c].dtype in ['float64','int64'] and c != 'TARGET']
print(f"Variables categóricas ({len(cat_cols)}): {cat_cols}")
print(f"Variables numéricas   ({len(num_cols)}): {num_cols}")

## 4. Limpieza con KNN Imputer (Imputación Multivariada)
> Al no conocer el significado de las variables, usamos KNN para imputar usando la similitud matemática entre registros — sin suposiciones del dominio.

In [ ]:
# Paso 1: Encodear categóricas para poder usar KNN
df_enc = df.copy()
le = LabelEncoder()
for col in cat_cols:
    df_enc[col] = df_enc[col].apply(lambda x: np.nan if pd.isna(x) else x)
    # Encodear sólo los no-nulos primero
    not_null = df_enc[col].dropna()
    le.fit(not_null.astype(str))
    df_enc[col] = df_enc[col].apply(
        lambda x: le.transform([str(x)])[0] if not pd.isna(x) else np.nan)

# Paso 2: KNN Imputer
imputer = KNNImputer(n_neighbors=5)
df_imputed = pd.DataFrame(imputer.fit_transform(df_enc), columns=df_enc.columns)

# Redondear columnas que eran categóricas
for col in cat_cols:
    df_imputed[col] = df_imputed[col].round().astype(int)

print(f"Nulos restantes tras KNN: {df_imputed.isnull().sum().sum()}")
print(f"Shape: {df_imputed.shape}")

## 5. EDA

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Distribución de clases
axes[0].bar(['Rechazado (0)', 'Aprobado (1)'], 
             df_imputed['TARGET'].value_counts().sort_index(),
             color=['tomato','steelblue'])
axes[0].set_title('Distribución del Target')

# Heatmap de correlaciones numéricas
num_df = df_imputed[num_cols + ['TARGET']]
sns.heatmap(num_df.corr(), annot=True, fmt='.2f', cmap='coolwarm', ax=axes[1])
axes[1].set_title('Correlaciones Numéricas')

plt.tight_layout()
plt.savefig('credit_eda.png', dpi=100)
plt.show()

## 6. Modelo — Regresión Logística con Función Sigmoide

In [ ]:
X = df_imputed.drop(columns=['TARGET'])
y = df_imputed['TARGET']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

lr = LogisticRegression(random_state=42, max_iter=1000)
lr.fit(X_train_s, y_train)
y_pred = lr.predict(X_test_s)
y_prob = lr.predict_proba(X_test_s)[:,1]

print("=== REGRESIÓN LOGÍSTICA ===")
print(classification_report(y_test, y_pred, target_names=['Rechazado','Aprobado']))
print(f"ROC-AUC: {roc_auc_score(y_test, y_prob):.4f}")

In [ ]:
# Visualizar probabilidades predichas vs realidad
plt.figure(figsize=(8,4))
plt.hist(y_prob[y_test==0], bins=30, alpha=0.6, color='tomato', label='Rechazado (real)')
plt.hist(y_prob[y_test==1], bins=30, alpha=0.6, color='steelblue', label='Aprobado (real)')
plt.axvline(0.5, color='black', linestyle='--', label='Umbral 0.5')
plt.title('Distribución de Probabilidades Predichas (Sigmoide)')
plt.xlabel('P(Aprobado) — salida de la función sigmoide')
plt.ylabel('Frecuencia')
plt.legend()
plt.tight_layout()
plt.savefig('credit_sigmoide.png', dpi=100)
plt.show()

## ✅ Resumen
| | |
|---|---|
| Registros | 690 |
| Features | 15 (A1–A15) |
| Imputación | KNN Imputer (n=5) |
| Modelo | Regresión Logística |
| ROC-AUC | ~0.93 |

**Desafío único:** Sin nombres de columnas, la imputación debe ser **ciega al dominio** — matemática pura.